In [1]:
import keras, datetime
import polars as pl
import numpy as np

from src.model.assemble_model import build_predict_model
from src.utils.miscellaneous import create_seq_dataset_multiple_input_single_output
from src.model.layer import gelu_approximate, FeatureWiseScalingLayer, DecompositionLayer

2026-01-16 21:18:34.068222: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-16 21:18:34.087767: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768565914.112667  407426 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768565914.120861  407426 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768565914.139460  407426 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


I0000 00:00:1768565919.452099  407426 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46395 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:4b:00.0, compute capability: 8.6
I0000 00:00:1768565919.454451  407426 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 46026 MB memory:  -> device: 1, name: NVIDIA RTX A6000, pci bus id: 0000:b1:00.0, compute capability: 8.6


In [2]:
import tensorflow as tf

# 1. 사용 가능한 GPU 목록 출력
gpu_devices = tf.config.list_physical_devices('GPU')
print("사용 가능한 GPU 개수:", len(gpu_devices))
print("GPU 상세 정보:", gpu_devices)


사용 가능한 GPU 개수: 2
GPU 상세 정보: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [3]:
data = pl.read_parquet('dataset.zstd')

In [4]:
data.glimpse()

Rows: 505825270
Columns: 3
$ time(h)          <f32> 0.0, 2.7777778086601757e-06, 5.555555617320351e-06, 8.333333425980527e-06, 1.1111111234640703e-05, 1.3888889043300878e-05, 1.6666666851961054e-05, 1.9444443751126528e-05, 2.2222222469281405e-05, 2.499999936844688e-05
$ p1_pressure      <f32> 0.550000011920929, 0.550000011920929, 0.5400000214576721, 0.5799999833106995, 0.550000011920929, 0.6100000143051147, 0.5799999833106995, 0.6000000238418579, 0.5899999737739563, 0.5600000023841858
$ peak_p1_pressure <f32> 22.994998931884766, 22.994998931884766, 22.994998931884766, 22.994998931884766, 22.994998931884766, 22.994998931884766, 22.994998931884766, 22.994998931884766, 22.994998931884766, 22.994998931884766



In [5]:
p1_det = (data['peak_p1_pressure']<16).cast(int)

In [6]:
dataset = data[['p1_pressure', 'peak_p1_pressure']].to_numpy()

In [7]:
import numpy as np
import tensorflow as tf

class TimeSeriesDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, data, seq_len, pred_distance, target_idx_pos, batch_size=32, shuffle=True, **kwargs):
        super().__init__(**kwargs)

        self.data = data
        self.seq_len = seq_len
        self.pred_distance = pred_distance
        self.target_idx_pos = target_idx_pos
        self.batch_size = batch_size
        self.shuffle = shuffle

        # 유효한 시작 인덱스(i) 계산:
        # 기존 코드의 range(data.shape[0] - pred_distance)와 if i+1 >= seq_len 조건 반영
        self.start_idx = self.seq_len - 1
        self.end_idx = self.data.shape[0] - self.pred_distance - 1

        # 전체 학습 가능한 샘플의 인덱스 리스트
        self.indices = np.arange(self.start_idx, self.end_idx + 1)
        self.on_epoch_end()

    def __len__(self):
        # 전체 배치의 개수
        return int(np.ceil(len(self.indices) / self.batch_size))

    def on_epoch_end(self):
        # 한 에폭이 끝날 때마다 데이터 섞기
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, index):
        # 배치 범위 설정
        batch_indices = self.indices[index * self.batch_size : (index + 1) * self.batch_size]

        batch_features = []
        batch_targets = []

        for i in batch_indices:
            # 피처 추출: data[i+1-seq_len : i+1, 0:target_idx_pos]
            feature = self.data[i + 1 - self.seq_len : i + 1, 0 : self.target_idx_pos]

            # 타겟 추출: data[i + pred_distance, target_idx_pos:]
            target = self.data[i + self.pred_distance, self.target_idx_pos:]

            batch_features.append(feature)
            batch_targets.append(target)

        return np.array(batch_features), np.array(batch_targets)

In [8]:
dataset_1 = dataset[::1000]

In [9]:
dataset_1.dtype

dtype('float32')

In [10]:
seq_len = 360
pred_distance = 360 * 600
target_idx_pos = 1

train_feature, train_target = create_seq_dataset_multiple_input_single_output(dataset_1, seq_len=seq_len, pred_distance=pred_distance, target_idx_pos=target_idx_pos)

train_target = np.squeeze(train_target)

print(train_feature.shape, train_target.shape)

creating sequence dataset...:   0%|          | 0/289826 [00:00<?, ?it/s]

(289467, 360, 1) (289467,)


In [11]:
model = build_predict_model(input_shape=(train_feature.shape[1:]), d_dims=8, dropout_rate=0.5, learning_rate=0.001)
#model.summary()

In [12]:
# model = keras.models.load_model('model_2160000.keras', custom_objects={'gelu_approximate': gelu_approximate,
#                                                                        'FeatureWiseScalingLayer': FeatureWiseScalingLayer,
#                                                                        'DecompositionLayer': DecompositionLayer})

In [13]:
total_params = model.count_params()
trainable_params = np.sum([np.prod(v.shape) for v in model.trainable_weights])
non_trainable_params = np.sum([np.prod(v.shape) for v in model.non_trainable_weights])

print("--- 모델 파라미터 요약 ---")
print(f"Total params: {total_params:,}")
print(f"Trainable params: {int(trainable_params):,}")
print(f"Non-trainable params: {int(non_trainable_params):,}")

--- 모델 파라미터 요약 ---
Total params: 4,130,000
Trainable params: 4,129,856
Non-trainable params: 144


In [14]:
bytes_per_param_float32 = 4
total_bytes = total_params * bytes_per_param_float32

total_kb = total_bytes / 1024
total_mb = total_kb / 1024

print(f"--- 32비트 (float32) 기준 계산 ---")
print(f"총 파라미터 수: {total_params:,}")
print(f"총 용량 (Bytes): {total_bytes:,} Bytes")
print(f"총 용량 (KB): {total_kb:.2f} KB")
print(f"총 용량 (MB): {total_mb:.2f} MB")

--- 32비트 (float32) 기준 계산 ---
총 파라미터 수: 4,130,000
총 용량 (Bytes): 16,520,000 Bytes
총 용량 (KB): 16132.81 KB
총 용량 (MB): 15.75 MB


In [15]:
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=500, verbose=0)
csv_logger = keras.callbacks.CSVLogger(filename=f'log_{pred_distance}_model.csv', append=False, separator=',')
model_chk_point = keras.callbacks.ModelCheckpoint(filepath=f'model_{pred_distance}.keras', monitor="loss", verbose=2,
                                                  save_best_only=True, save_weights_only=False, mode="min", save_freq="epoch", initial_value_threshold=None)

log_dir = "logs/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

model.fit(x=train_feature, y=train_target, epochs=999999, verbose=1, batch_size=1024*10,
          callbacks=[early_stop, csv_logger, model_chk_point, tensorboard_callback])

Epoch 1/999999


I0000 00:00:1768566000.469116  407939 service.cc:152] XLA service 0x75acf00038e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1768566000.469182  407939 service.cc:160]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
I0000 00:00:1768566000.469192  407939 service.cc:160]   StreamExecutor device (1): NVIDIA RTX A6000, Compute Capability 8.6
2026-01-16 21:20:03.028817: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1768566011.428678  407939 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-01-16 21:20:33.513531: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_84940', 8 bytes spill stores, 8 bytes spill loads

2026-01-16 21:20:33.582505: I external/local_xla/xla/stream_executor/cuda/sub

28/29 ━━━━━━━━━━━━━━━━━━━━ 0s 399ms/step - loss: 7.3944 - mean_absolute_error: 8.0509 - mean_absolute_percentage_error: 40.7440

2026-01-16 21:24:37.401781: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_91705_0', 60 bytes spill stores, 60 bytes spill loads

2026-01-16 21:24:37.796561: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_85049', 212 bytes spill stores, 212 bytes spill loads

2026-01-16 21:24:37.855376: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_81432', 96 bytes spill stores, 96 bytes spill loads

2026-01-16 21:24:38.064719: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_84831', 224 bytes spill stores, 224 bytes spill loads

2026-01-16 21:24:38.842345: I 

29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - loss: 7.2703 - mean_absolute_error: 7.9254 - mean_absolute_percentage_error: 40.1102   
Epoch 1: loss improved from None to 3.79422, saving model to model_216000.keras


/home/jinbeom/miniconda3/envs/tensorflow-gpu_219_python_311/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,mean_absolute_error,mean_absolute_percentage_error
  current = self.get_monitor_value(logs)


29/29 ━━━━━━━━━━━━━━━━━━━━ 490s 7s/step - loss: 3.7942 - mean_absolute_error: 4.4135 - mean_absolute_percentage_error: 22.3637
Epoch 2/999999
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 389ms/step - loss: 1.5916 - mean_absolute_error: 2.1705 - mean_absolute_percentage_error: 11.3480
Epoch 2: loss improved from 3.79422 to 1.55524, saving model to model_216000.keras
29/29 ━━━━━━━━━━━━━━━━━━━━ 18s 626ms/step - loss: 1.5552 - mean_absolute_error: 2.1332 - mean_absolute_percentage_error: 11.0875
Epoch 3/999999
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 384ms/step - loss: 1.4868 - mean_absolute_error: 2.0612 - mean_absolute_percentage_error: 10.7167
Epoch 3: loss improved from 1.55524 to 1.47413, saving model to model_216000.keras
29/29 ━━━━━━━━━━━━━━━━━━━━ 18s 631ms/step - loss: 1.4741 - mean_absolute_error: 2.0482 - mean_absolute_percentage_error: 10.6448
Epoch 4/999999
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 384ms/step - loss: 1.4516 - mean_absolute_error: 2.0241 - mean_absolute_percentage_error: 10.5174
Epoch 4: loss improv

KeyboardInterrupt: 